# Lesson 7 — Trees and Ensembles

Self-assessment. No code: every answer is a sentence, a short calculation, or a
diagnosis.

Numbers quoted throughout come from the lesson's handout and notebooks: 1,200
loan applicants, 38.7% of them defaulted (464 of 1,200), so the majority
baseline is 0.613. 7% of labels are deliberately flipped, which sets a noise
ceiling of about 0.93 (1 minus the flip rate) — no model on this data should
be expected to reach it, and none should be trusted if it does. As in earlier
lessons, several questions describe a situation and ask you to *criticise* or
*derive* it; those are the ones worth your time.

## Part 1 — Decision trees and Gini impurity

**1. State the whole decision tree algorithm in your own words, and explain precisely what "greedy" means for how it searches.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Find the single (feature, threshold) split that makes the two resulting groups as pure as possible, apply it, then repeat the entire search independently on each resulting group until a stopping rule is met.</li>
        <li>"Greedy" means at every step the tree takes whichever split maximises the impurity reduction &Delta;G <b>right now</b>, with no mechanism for looking ahead to a split that would help more two levels down &mdash; a feature that only pays off in combination with a later split can be invisible to the search.</li>
        <li>Nothing is <i>estimated</i> the way lessons 3 and 4 used the word &mdash; there is no coefficient and no gradient with respect to a parameter vector; each split is chosen once, by exhaustive search, and never revisited.</li>
    </ul>
    </p>
</details>

**2. Define the Gini impurity <code>G</code> of a group of examples drawn from <code>C</code> classes, and explain precisely what it measures.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>G = 1 - &Sigma;<sub>c</sub> p<sub>c</sub><sup>2</sup></code>, where <code>p<sub>c</sub></code> is the fraction of the group belonging to class <code>c</code>.</li>
        <li>It is <b>not</b> a probability of misclassification. It is the probability that two examples drawn independently, at random and <b>with replacement</b> from the group, and labelled according to the group's own class frequencies, would <b>disagree</b>.</li>
        <li>For two classes it simplifies to <code>G = 2p(1-p)</code>: <b>0</b> when the group is pure (<code>p &isin; {0, 1}</code>) and maximal, <b>0.5</b>, when <code>p = 0.5</code> &mdash; the group is as mixed as it can be.</li>
    </ul>
    </p>
</details>

**3. The loan dataset has 464 of 1,200 applicants defaulting, so p = 0.387 at the root. Using the two-class Gini formula, show that G(parent) &asymp; 0.474, and name the split the handout reports as the one that maximises &Delta;G at the root.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Two-class formula: <code>G = 2p(1-p) = 2 &times; 0.387 &times; 0.613</code>, using <code>1 - p = 0.613</code> for the repaid class.</li>
        <li>Arithmetic: <code>0.387 &times; 0.613 &asymp; 0.2372</code>, doubled gives <code>&asymp; 0.4743</code> &mdash; matching the handout's quoted 0.474 to three decimal places.</li>
        <li>The best root split found by exhaustive search is <code>debt_ratio &lt;= 0.82</code> &mdash; recognisably the debt ceiling the data was generated with.</li>
        <li>The from-scratch implementation confirms this matches scikit-learn's <code>DecisionTreeClassifier</code> to the fourth decimal place of resulting accuracy, at every depth tested.</li>
    </ul>
    </p>
</details>

## Part 2 — Depth and the bias-variance trade-off

**4. Compare the training accuracy of a depth-2 tree with that of an unconstrained tree, and explain precisely what a training accuracy of 1.000 demonstrates about the model.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A depth-2 tree reaches only 0.840 training accuracy, because it can afford only the coarsest structure &mdash; the income floor and the debt ceiling.</li>
        <li>An unconstrained tree reaches training accuracy <b>1.000</b> &mdash; on <i>any</i> dataset whatsoever, including one with no structure at all: keep splitting and every leaf eventually holds one example, which it then "predicts" perfectly.</li>
        <li>1.000 demonstrates that the model class is flexible enough to <b>memorise</b> the training set, not that anything general has been learned &mdash; the same reading lesson 6 gave k = 1 nearest neighbours' training accuracy of 1.000.</li>
    </ul>
    </p>
</details>

**5. Cross-validated accuracy peaks at depth 8 (0.882) while the unconstrained tree reaches training accuracy 1.000 but only 0.852 cross-validated. Describe the shape of this curve, and explain where the "extra" splits past depth 8 are actually spent.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Training accuracy climbs monotonically towards 1.000 as depth grows, but cross-validated accuracy rises, peaks at depth 8, and then falls &mdash; even though the model keeps getting more flexible.</li>
        <li>Past the peak, every extra split is spent fitting some of the dataset's 7% deliberately flipped labels, which have no pattern to learn; a tree that classifies a flipped label "correctly" on the training set has memorised it, not generalised it.</li>
        <li>The unconstrained tree loses <b>3 points of cross-validated accuracy</b> relative to the depth-8 peak (0.852 vs 0.882) while gaining nothing but a training score of 1.000 that nobody should have trusted in the first place.</li>
    </ul>
    </p>
</details>

**6. It is tempting to expect a more flexible model to do at least as well as a less flexible one on held-out data &mdash; more options can only help, surely. State precisely where this instinct is correct and where it fails.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The instinct is sound for <b>bias</b>: a deeper tree can represent anything a shallower one can, plus more, so its best-case fit to the true rule can only improve or stay the same.</li>
        <li>It is wrong for the quantity actually measured (cross-validated accuracy), because flexibility increases <b>variance</b> at the same time, and past some point the added variance costs more than the reduced bias saves.</li>
        <li>This is the second time this course has shown the curve bend downward &mdash; lesson 3's Lasso penalty was the first &mdash; and depth is trees' version of the same dial that k was for k-nearest neighbours in lesson 6.</li>
    </ul>
    </p>
</details>

## Part 3 — Tree readability and instability

**7. What makes a single decision tree unusual among the models covered so far in this course, and how does growing it with <code>max_leaf_nodes</code> rather than <code>max_depth</code> serve that property?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A trained tree is a sequence of if/else statements &mdash; the entire model can be printed and read end to end by a person without a mathematics background, which k-nearest neighbours and support vector machines cannot offer at all.</li>
        <li><code>max_depth</code> grows the tree <b>breadth-first</b>: every leaf at level d is split before any leaf reaches level d+1, whether or not that split is worth having. <code>max_leaf_nodes</code> grows it <b>best-first</b> instead, always expanding whichever leaf offers the largest &Delta;G.</li>
        <li>With <code>max_leaf_nodes = 9</code> the loan tree reaches 90.8% training accuracy in nine leaves, and every threshold is recognisable from the generating rule &mdash; a compact, readable model, rather than one that spent splits on low-value questions just because a depth budget allowed it.</li>
    </ul>
    </p>
</details>

**8. Two trees at max_depth = 6, each fit to an independent 65% resample of the same 1,200 applicants, find the debt ceiling within 0.001 of each other, yet disagree on 11.7% of predictions and grow 31 and 33 leaves respectively. Explain why the strongest split is stable while the tree as a whole is not.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The debt ceiling is the strongest split: the evidence for it is overwhelming and spread across most of the data, so it barely moves between resamples &mdash; hence agreement to within 0.001.</li>
        <li>The weaker splits further down are each supported by far fewer points, so a resample that happens to include or exclude a handful of borderline rows can change which threshold looks best there &mdash; and those weaker splits are most of what determines a tree's final shape.</li>
        <li>The result is a tree that agrees almost exactly on the one question backed by overwhelming evidence and disagrees substantially wherever the evidence is thinner, which is enough to produce 11.7% disagreement on overall predictions despite near-identical root splits.</li>
    </ul>
    </p>
</details>

## Part 4 — Bagging and the bootstrap

**9. State the bagging (bootstrap aggregating) algorithm precisely, and say what part of the tree algorithm from Part 1 bagging changes.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Draw a <b>bootstrap sample</b>: m rows chosen <b>with replacement</b> from the m training rows, so some rows appear more than once and some not at all. Fit a tree to it. Repeat <code>n_estimators</code> times. To predict, ask every tree and take the majority vote.</li>
        <li>Bagging changes nothing about how any individual tree is grown &mdash; it reuses the Gini-impurity splitting algorithm from Part 1 unmodified &mdash; and changes only what surrounds it: which rows each tree sees, and how the trees' predictions are combined.</li>
    </ul>
    </p>
</details>

**10. Derive, from the definition of a bootstrap sample, why a single sample leaves roughly 36.8% of the original rows out entirely as the number of rows m grows large, and state what that left-out set is called.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Each of the m draws independently avoids one specific row with probability <code>(1 - 1/m)</code>, and the row is drawn with replacement m times, so the chance that specific row is never drawn across all m draws is <code>(1 - 1/m)<sup>m</sup></code>.</li>
        <li>As <code>m &rarr; &infin;</code>, this is the standard exponential limit <code>lim (1 + x/m)<sup>m</sup> = e<sup>x</sup></code> with <code>x = -1</code>, so <code>(1 - 1/m)<sup>m</sup> &rarr; e<sup>-1</sup> &asymp; 0.368</code>.</li>
        <li>Each bootstrap sample therefore contains about <code>1 - e<sup>-1</sup> &asymp; 63.2%</code> of the distinct rows and leaves about <b>36.8%</b> out entirely; the left-out rows are called <b>out-of-bag (OOB)</b>. At m = 1,200 the finite-sample value <code>(1 - 1/1200)<sup>1200</sup> = 0.3677</code> is already within 0.0002 of the limit.</li>
    </ul>
    </p>
</details>

**11. On the loan data, a 300-tree random forest's out-of-bag (OOB) score was 0.9117 and its 5-fold cross-validated accuracy was 0.9117 &plusmn; 0.021 — the same to four decimal places. Why should these two very differently computed numbers agree, and how much agreement should you actually expect?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Every OOB row is free validation data for the one tree that did not see it during training; averaging each tree's OOB predictions gives the OOB score, at no extra split of the data.</li>
        <li>OOB score and cross-validation are estimating the <b>same quantity</b> &mdash; how well the ensemble generalises to unseen rows &mdash; by different bookkeeping: OOB score has each tree supply its own held-out fold, while cross-validation splits the whole dataset up front.</li>
        <li>How much agreement to expect is <b>to within their own noise</b>, not to every decimal. Repeating both over twelve seeds gives a typical gap of <code>0.0026</code> and a worst case of <code>0.0075</code>, against a cross-validation spread of &plusmn;0.021 &mdash; so the exact four-decimal match on this seed is a coincidence, and reading it as the expected behaviour would be over-reading a single run.</li>
    </ul>
    </p>
</details>

**12. Across 30 independent 70/30 train/test splits, a single unconstrained tree's test accuracy has mean 0.856 (standard deviation 0.0185); a 100-tree bagged ensemble has mean 0.902 (standard deviation 0.0153). Explain why both numbers moved in the observed direction.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The mean rose because bagging also averages out some of a single deep tree's overfitting &mdash; each tree in the ensemble still overfits its own bootstrap sample, but their individual errors are not perfectly correlated, so the majority vote corrects some of them.</li>
        <li>The standard deviation fell because averaging correlated estimators reduces variance, exactly as the variance formula in Part 5 predicts: more trees pull the ensemble's variance down towards a floor set by how correlated the trees are, not to zero.</li>
        <li>Both changes are two faces of the same mechanism &mdash; averaging trees whose mistakes are not identical &mdash; not two unrelated effects of adding more trees.</li>
    </ul>
    </p>
</details>

## Part 5 — Why averaging reduces variance

**13. Starting from Var((1/B) &Sigma; f<sub>b</sub>) for B trees, each of variance &sigma;<sup>2</sup> and average pairwise correlation &rho;, derive the expression the handout gives for the variance of the averaged prediction, and state its limit as B &rarr; &infin;.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Expanding the sum: <code>Var((1/B)&Sigma;f<sub>b</sub>) = (1/B<sup>2</sup>)(&Sigma;Var(f<sub>b</sub>) + &Sigma;<sub>b&ne;b'</sub>Cov(f<sub>b</sub>,f<sub>b'</sub>))</code>. There are B variance terms, each &sigma;<sup>2</sup>, and B(B-1) covariance terms, each &rho;&sigma;<sup>2</sup>.</li>
        <li>Summing: <code>(1/B<sup>2</sup>)(B&sigma;<sup>2</sup> + B(B-1)&rho;&sigma;<sup>2</sup>) = &sigma;<sup>2</sup>/B + ((B-1)/B)&rho;&sigma;<sup>2</sup></code>.</li>
        <li>As <code>B &rarr; &infin;</code>, <code>&sigma;<sup>2</sup>/B &rarr; 0</code> and <code>(B-1)/B &rarr; 1</code>, so the variance tends to <b>&rho;&sigma;<sup>2</sup></b> &mdash; a floor, not zero. Adding trees past the point where <code>&sigma;<sup>2</sup>/B</code> is already small buys almost nothing, because the <code>&rho;&sigma;<sup>2</sup></code> term does not shrink with B at all.</li>
    </ul>
    </p>
</details>

**14. The handout calls bagging "a variance tool" that "does nothing for a tree that is too shallow to represent the rule in the first place." Explain why, and predict roughly what would happen if 100 depth-2 trees were bagged on the loan data.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Averaging unbiased-but-noisy trees gives an unbiased average; averaging trees that share a <b>systematic</b> error preserves that error exactly, because the error is identical across all the trees being averaged rather than random noise that partly cancels.</li>
        <li>A depth-2 tree cannot represent the two interior stressed islands at all (Part 1 established that neither island is visible at depth 2) &mdash; that is a <b>bias</b> problem, not a variance problem, and it is the same for every depth-2 tree regardless of which bootstrap sample it saw.</li>
        <li>Bagging 100 such trees would tighten the spread around roughly the same, still-low accuracy a single depth-2 tree gets. It would not close the gap to a properly depth-tuned tree, because averaging cannot fix a hypothesis class that is too limited to represent the target.</li>
    </ul>
    </p>
</details>

## Part 6 — Random forests and feature importance

**15. What single restriction distinguishes a random forest from plain bagging, and why does it lower &rho; (the correlation between trees) further than bootstrap resampling alone does?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>At each split, only a random subset of <code>max_features</code> features &mdash; by default &lfloor;&radic;n&rfloor; for classification &mdash; is even offered as a candidate, rather than every feature.</li>
        <li>Bagged trees still consider every feature at every split, so a strong feature pulls most bootstrap trees toward splitting on it first regardless of which rows they drew, which keeps their predictions correlated.</li>
        <li>Forcing different trees to consider different features at the same point in the tree makes them genuinely disagree more often, lowering &rho; and letting the variance floor &rho;&sigma;<sup>2</sup> drop further &mdash; without changing the splitting rule itself.</li>
    </ul>
    </p>
</details>

**16. With <code>max_features="sqrt"</code> and 22 total columns, each split considers &lfloor;&radic;22&rfloor; = 4 candidate features. Compute the probability that one specific fixed feature is offered as a candidate at a given split, and the probability it is excluded. Explain why this matters for pure-noise columns.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>By symmetry, the number of size-k subsets of p features that contain one fixed feature is <code>C(p-1, k-1)</code> against <code>C(p, k)</code> subsets in total, so the probability that specific feature is a candidate is <code>C(p-1,k-1)/C(p,k) = k/p = 4/22 &asymp; 18.2%</code>.</li>
        <li>The probability it is <b>excluded</b> from a given split is therefore <code>&asymp; 81.8%</code> &mdash; roughly four times out of five, neither real feature is even offered as a candidate.</li>
        <li>When a real feature is excluded, the tree must choose its best split among whichever noise columns happened to be drawn; among 1,200 finite, noisy rows some noise column will show a nonzero &Delta;G purely by chance, every time. Each such gain is small, but accumulated across hundreds of trees and thousands of splits it produces a non-negligible share of total importance.</li>
    </ul>
    </p>
</details>

**17. With 20 pure-noise columns added to the loan data's 2 real features, 54.5% of the random forest's total feature importance landed on noise, and cross-validated accuracy fell to 0.837. State this result precisely, explain why it does not contradict Part 5's variance-reduction result, and give the practical remedy the handout suggests.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With 22 total columns (2 real, 20 pure noise, each <code>np.random.normal</code> and wired to nothing), <b>54.5% of the forest's summed impurity-reduction importance sits on the noise columns</b>, and cross-validated accuracy drops from 0.912 (no noise columns) to 0.837.</li>
        <li>Averaging genuinely reduces the variance of the ensemble's <i>predictions</i> &mdash; that result is real. It says nothing about the expected importance assigned to any one feature: the same feature-subsampling restriction that decorrelates the trees is what occasionally hands a split to noise, at &asymp; 81.8% exclusion odds for any one real feature.</li>
        <li>The remedy: treat a random forest's importances as <b>suggestive, not definitive</b> &mdash; cross-check against permutation importance on held-out data, or against which columns could plausibly matter before asking the forest which ones do.</li>
    </ul>
    </p>
</details>

## Part 7 — Gradient boosting

**18. Write the boosting update rule F<sub>t</sub>(x) = F<sub>t-1</sub>(x) + &alpha;&middot;h<sub>t</sub>(x), and define each term, including what distinguishes h<sub>t</sub> from the trees built in Parts 4 and 6.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>F<sub>0</sub>(x)</code> is initialised to the mean target, <code>&#563;</code>; each subsequent round adds <code>&alpha;</code> times a newly fit tree's output to the running prediction.</li>
        <li><code>h<sub>t</sub></code> is a shallow tree &mdash; depth 2 or 3, a <b>weak learner</b> &mdash; fit specifically to what remains unexplained by <code>F<sub>t-1</sub></code>, not to the original target directly.</li>
        <li><code>&alpha;</code> is the <b>learning rate</b>: how much of each new tree's correction is actually applied &mdash; the same symbol lesson 3 used for gradient descent's step size. Unlike bagging and random forests, which build trees <b>independently</b> and average at the end, boosting builds trees <b>in sequence</b>, each correcting what came before.</li>
    </ul>
    </p>
</details>

**19. For squared-error loss L(y, F) = &frac12;(y - F)&sup2;, derive -&part;L/&part;F and explain why "fit the residual" and "fit the negative gradient" coincide exactly for this loss. Then, for classification with log-loss and p = &sigma;(F), state the pseudo-residual the handout derives and what earlier quantity it matches.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Squared error: <code>&part;L/&part;F = -(y - F) = F - y</code>, so the negative gradient is <code>-&part;L/&part;F = y - F</code> &mdash; exactly the residual, the true target minus the current prediction. "Fit the negative gradient" (the general functional-gradient-descent recipe) is therefore literally the same computation as "fit the residual" for this one loss.</li>
        <li>Classification: by the chain rule, <code>&part;L/&part;F = (&part;L/&part;p)(&part;p/&part;F) = (-y/p + (1-y)/(1-p)) &middot; p(1-p) = p - y</code>, using the logistic function's own derivative <code>p(1-p)</code>.</li>
        <li>The negative gradient &mdash; the <b>pseudo-residual</b> each tree is fit to &mdash; is <code>y - p</code>: the true label minus the current predicted probability, exactly the error term lesson 4's logistic regression gradient descent moved against.</li>
    </ul>
    </p>
</details>

**20. In the worked example (a step function plus a gentle sine wave, fit by depth-2 trees at &alpha; = 0.3), mean squared error (MSE) against the true function was 0.395 after 1 tree, 0.068 after 5, 0.012 after 20, and 0.018 after 60. Explain what this progression demonstrates.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>One tree barely moves the flat starting guess (MSE 0.395); five trees sketch the coarse shape (MSE 0.068); twenty trees trace the step function closely, oscillation included, at the ensemble's best point (MSE 0.012).</li>
        <li>By sixty trees the ensemble is tracking individual noisy observations as well as the underlying shape, and the error against the noise-free truth has risen back to 0.018 &mdash; worse than at twenty trees despite continuing to add capacity.</li>
        <li>This is the same overfitting story as the unconstrained single tree from Part 2, reached by a different route: not one tree grown too deep, but too many shallow trees each eventually chasing whatever residual noise is left rather than remaining signal.</li>
    </ul>
    </p>
</details>

## Part 8 — Overfitting in boosting

**21. Why does boosting keep reducing training error for as long as it runs, unlike bagging, and what does this imply once the 7% deliberately flipped labels are the only thing left to explain?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Bagging averages independently grown trees, each fit once to its own bootstrap sample, with no mechanism that keeps forcing the ensemble to explain more of the training set as more trees are added.</li>
        <li>Boosting builds each new tree specifically to correct whatever the ensemble so far got wrong, so as long as any training error remains, the next tree has something to fit &mdash; including noise, once the genuine signal is exhausted.</li>
        <li>With nothing to stop it, that includes the 7% of labels that are noise by construction: training accuracy on the loan data reaches 1.000 by 200 trees and stays there, well past the point where cross-validated accuracy has already peaked and started to fall.</li>
    </ul>
    </p>
</details>

**22. An unconstrained single tree loses about 3 points of cross-validated accuracy past its peak (0.882 to 0.852); 800 boosted trees lose about 1 point past their peak near 30 trees, even with training accuracy pinned at 1.000 since well before that. Explain why the handout calls boosting's overfitting "gentler," and why that makes it more dangerous to miss rather than less.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Boosting's overfitting is real &mdash; cross-validated accuracy does drift down after the peak near 30 trees &mdash; but the loss per added tree is small: about 1 point spread over roughly 770 additional trees, versus the single tree's 3-point loss reached in effectively one step.</li>
        <li>That gentleness is exactly why early stopping matters for boosting rather than being optional: the damage per added tree is small and easy to miss on a single run, but it is still there and it accumulates.</li>
        <li>A practitioner watching only the training curve (pinned at 1.000 for hundreds of trees) would see no warning sign at all; only the cross-validated curve shows the slow decline.</li>
    </ul>
    </p>
</details>

**23. Fixing <code>n_estimators = 120</code> and varying only the learning rate &alpha;: &alpha; = 0.02 gives training accuracy 0.888, &alpha; = 1.00 gives training accuracy 1.000 with cross-validated accuracy down to 0.878, and &alpha; = 0.10 peaks at 0.902 cross-validated. Explain why learning rate and tree count are not independent knobs.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Both &alpha; and <code>n_estimators</code> control how much total correction the ensemble applies &mdash; a large &alpha; over few trees can apply as much net correction as a small &alpha; over many trees &mdash; so they trade off against each other rather than acting as separate, independent dials.</li>
        <li>&alpha; = 0.02 has not finished fitting the signal within 120 trees (training accuracy still under 0.89); &alpha; = 1.00 has memorised the training set completely (1.000) and given back several points of cross-validated accuracy for it.</li>
        <li>The best setting sits in between: large enough to make real progress within the tree budget, small enough that each individual correction is easy to outvote by later trees if it turns out to be wrong &mdash; which is why the two hyperparameters must be tuned together, not one at a time.</li>
    </ul>
    </p>
</details>

## Part 9 — Choosing among trees, forests and boosting

**24. On the full leaderboard, every ensemble method (bagging, random forest, gradient boosting) lands within about a point and a half of the others, all closer to the noise ceiling than either single tree. State the practical conclusion the handout draws from this, and contrast it with the 3-point gap between the unconstrained tree and the depth-tuned one.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The practical conclusion: which ensembling <b>strategy</b> you pick (bagging vs random forest vs boosting) matters far less than <b>whether you ensemble at all</b> &mdash; the three cluster together (0.898 to 0.911) while both single trees sit further below.</li>
        <li>The unconstrained tree (0.852) and the depth-tuned single tree (0.883) differ by about 3 points from each other despite being the same family &mdash; a bigger swing than the gap between the best and worst ensemble method.</li>
        <li>So depth-tuning a single tree buys more than switching among untuned ensembles would, but no single tree, however well tuned, reaches where the ensembles land.</li>
    </ul>
    </p>
</details>

**25. A colleague needs a model whose decision can be explained, point by point, to the person it affects, and wants minimal ongoing engineering effort. Which method should they reach for, and which should they avoid even though it scores higher on this data?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Reach for a <b>single tree, depth-tuned</b> (or nothing on this list, if even that is unacceptable): it is the only model on the leaderboard a person could read start to finish, a sequence of if/else statements rather than hundreds of trees voting.</li>
        <li>Avoid the <b>random forest</b>, despite its higher accuracy (0.911 vs the depth-tuned tree's 0.883): its prediction is the combination of many trees' votes, which cannot be read out as a simple justification the way one tree's path can.</li>
        <li>The accuracy cost of choosing the tree is small in absolute terms (about 3 points, and both are well short of the 0.93 noise ceiling anyway) against a real gain in being able to explain the decision.</li>
    </ul>
    </p>
</details>